In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

#パスの設定
input_path = Path("/content/drive/MyDrive/因果推論/h3_mesh_panel.csv")
input_path.parent.mkdir(parents=True, exist_ok=True)
output_dir = input_path.parent

#介入日の設定
intervention_date = pd.Timestamp("2025-01-01")
#商圏重複率の除外基準を設定
overlap_threshold = 0.60
#データの読み込む
df = pd.read_csv(input_path,parse_dates=["date"])

#必要な列が揃っているか確認
required_columns = {"date","h3_id","visitors","treated"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"必要な列がありません: {sorted(missing_columns)}")

#重複行ないか確認
if df.duplicated(["date", "h3_id"]).any():
    raise ValueError("date × h3_id に重複があります。")

#型を指定
df["treated"] = df["treated"].astype(int)
df["visitors"] = pd.to_numeric(df["visitors"],errors="raise")

#メッシュIDを確認
all_mesh_ids = sorted(df["h3_id"].unique())
treated_mesh_ids = sorted(df.loc[df["treated"] == 1,"h3_id"].unique())
control_mesh_ids = sorted(df.loc[df["treated"] == 0,"h3_id"].unique())
print("Number of all meshes:", len(all_mesh_ids))
print("Number of treated meshes:", len(treated_mesh_ids))
print("Number of control meshes:", len(control_mesh_ids))

#h3_id × station の対応表を作成
# h3_mesh_01 ～ h3_mesh_06 → shibuya
# h3_mesh_07 ～ h3_mesh_12 → ikebukuro
# h3_mesh_13 ～ h3_mesh_18 → ueno
# h3_mesh_19 ～ h3_mesh_24 → kawasaki
# h3_mesh_25 ～ h3_mesh_30 → omiya
# h3_mesh_31 ～ h3_mesh_36 → kichijoji

mesh_station_mapping = pd.DataFrame({
    "h3_id": [f"h3_mesh_{i:02d}" for i in range(1, 37)],
    "station": (["shibuya"] * 6 + ["ikebukuro"] * 6 + ["ueno"] * 6 + ["kawasaki"] * 6 + ["omiya"] * 6 + ["kichijoji"] * 6)})

#データフレームにstation列を追加
df = df.merge(mesh_station_mapping,on="h3_id",how="left",validate="many_to_one")

#整合性を確認
if df["station"].isna().any():
    raise ValueError("対応しないメッシュがあります。")

expected_treated = df["station"].eq("shibuya").astype(int)
if not np.array_equal(df["treated"].to_numpy(),expected_treated.to_numpy()):
    raise ValueError("stationとtreatedの対応が矛盾しています。")

#教材用に商圏重複スコアを定義（人工的に設定した値をJaccard係数として使用）
station_overlap_score = {
    "ikebukuro": 0.75,  #渋谷と池袋の来訪者集合のJaccard係数が0.75
    "ueno": 0.45,
    "kawasaki": 0.25,
    "omiya": 0.20,
    "kichijoji": 0.65,
}
#ドナーメッシュのメタデータ作成
donor_metadata = mesh_station_mapping.loc[mesh_station_mapping["station"] != "shibuya"].copy()
donor_metadata["overlap_score"] = donor_metadata["station"].map(station_overlap_score)
if donor_metadata["overlap_score"].isna().any():
    raise ValueError("overlap_scoreが設定されていない駅があります。")
donor_metadata["excluded_high_overlap"] = (donor_metadata["overlap_score"] >= overlap_threshold)

#除外するドナー
excluded_donors = donor_metadata.loc[donor_metadata["excluded_high_overlap"],"h3_id"].tolist()
#使用するドナー
eligible_donors = donor_metadata.loc[~donor_metadata["excluded_high_overlap"],"h3_id"].tolist()
#全てのドナー
all_donors = donor_metadata["h3_id"].tolist()

if len(eligible_donors) < 2:
    raise ValueError("除外後のドナー数が不足しています。")

#処置エリアの平均系列
#6つの渋谷メッシュの平均を1つの処置エリア系列とする。
treated_series = (df.loc[df["station"] == "shibuya"].groupby("date")["visitors"]
    .mean().sort_index())
treated_series.name = "actual_treated_area"

#ドナーメッシュを横持ちにする
donor_wide = (df.loc[df["station"] != "shibuya"].pivot(index="date",columns="h3_id",
        values="visitors").sort_index())
donor_wide = donor_wide.loc[:,all_donors]

#Pre / Post
pre_mask = treated_series.index < intervention_date
post_mask = treated_series.index >= intervention_date

#全てのドナーを使った場合の反実仮想と高重複ドナー除外後の仮想現実を作り比較するため、Synthetic Controlの関数を作成
def fit_scm(donor_units):
    #介入前の処置系列を作成
    y_pre = treated_series.loc[pre_mask].to_numpy(dtype=float)
    #介入前のドナー系列を作成
    x_pre = donor_wide.loc[pre_mask,donor_units].to_numpy(dtype=float)
    #エラーハンドリング
    if np.var(y_pre) == 0:
        raise ValueError("施策前の処置系列の分散が0です。")
    #重みを推定を目的とした最適化に代入する目的関数を定義
    def objective(weights):
        #期間全体で反実仮想を作成
        synthetic_pre = (x_pre @ weights)
        #観測値と反実仮想のGapを抽出
        residual = y_pre - synthetic_pre
        #mse/観測値の分散を計算
        return np.mean(residual ** 2) / np.var(y_pre)

    #最適化を実行し結果を格納
    #重みの初期値は全部同じ値になるように設定
    n_donors = len(donor_units)
    result = minimize(objective,x0=np.full(n_donors,1.0 / n_donors),
        method="SLSQP",bounds=[(0.0, 1.0) for _ in range(n_donors)],
        constraints={
            "type": "eq",
            "fun":lambda weights:weights.sum() - 1.0},
        options={
            "ftol": 1e-12,
            "maxiter": 5000,
            "disp": False,
        })
    #エラーハンドリング
    if not result.success:
        raise RuntimeError(f"SCM重み推定に失敗しました。\n {result.message}")
    #最適化で推定した重みを格納
    weights = pd.Series(result.x,index=donor_units,name="weight")
    #数値誤差レベルの重みを0へ
    weights[np.abs(weights) < 1e-8] = 0.0
    #反実仮想を作成
    synthetic = pd.Series(donor_wide[donor_units].to_numpy(dtype=float) @ weights.to_numpy(),
        index=donor_wide.index,name="synthetic")
    #観測値と反実仮想のGapを抽出
    gap = treated_series - synthetic
    pre_gap = gap.loc[pre_mask].to_numpy(dtype=float)
    post_gap = gap.loc[post_mask].to_numpy(dtype=float)
    #介入前後のRMSPEを計算
    pre_rmspe = np.sqrt(np.mean(pre_gap ** 2))
    post_rmspe = np.sqrt(np.mean(post_gap ** 2))
    #エラーハンドリング
    if np.isclose(pre_rmspe,0.0):
        raise ValueError("Pre RMSPEが0のためRatioを計算できません。")
    return {
        "weights": weights,
        "synthetic": synthetic,
        "gap": gap,
        "pre_rmspe": pre_rmspe,
        "post_rmspe": post_rmspe,
        "rmspe_ratio":post_rmspe / pre_rmspe,
        "mean_post_gap":gap.loc[post_mask].mean()
    }

#除外前SCM
result_all = fit_scm(all_donors)
#高重複ドナー除外後SCM
result_excluded = fit_scm(eligible_donors)
#真の施策効果
if ("treatment_effect_true" in df.columns):
    true_effect = (df.loc[(df["treated"] == 1) & (df["date"] >= intervention_date),
            "treatment_effect_true"].mean())
else:
    true_effect = np.nan

#除外前後の結果比較
comparison_df = pd.DataFrame({
    "model": ["All donors","High-overlap donors excluded"],
    "donor_count": [len(all_donors),len(eligible_donors)],
    "pre_rmspe": [result_all["pre_rmspe"],result_excluded["pre_rmspe"]],
    "post_rmspe": [result_all["post_rmspe"],result_excluded["post_rmspe"]],
    "post_pre_rmspe_ratio": [result_all["rmspe_ratio"],result_excluded["rmspe_ratio"]],
    "mean_post_gap": [result_all["mean_post_gap"],result_excluded["mean_post_gap"]],
    "true_effect": [true_effect,true_effect]})

comparison_df["estimation_error"] = (comparison_df["mean_post_gap"] - comparison_df["true_effect"])
display(comparison_df)

#処置群、除外前ドナー群、除外後ドナー群、除外前ドナー群とのGAP、除外後ドナー群のGAPの日次系列
series_df = pd.DataFrame({
    "actual_treated_area":treated_series,
    "synthetic_all_donors":result_all["synthetic"],
    "synthetic_overlap_excluded":result_excluded["synthetic"],
    "gap_all_donors":result_all["gap"],
    "gap_overlap_excluded":result_excluded["gap"]})
display(series_df.head())

#どのドナーメッシュが合成渋谷の形成に最も貢献しているかを見るため重みの大きさを表で確認
weights_before_df = (result_all["weights"].rename_axis("h3_id").reset_index()
    .merge(donor_metadata,on="h3_id",how="left",validate="one_to_one")
    .sort_values("weight",ascending=False).reset_index(drop=True))
display(weights_before_df.head())

weights_after_df = (result_excluded["weights"].rename_axis("h3_id").reset_index()
    .merge(donor_metadata[["h3_id","station","overlap_score"]],on="h3_id",
    how="left",validate="one_to_one").sort_values("weight",ascending=False).reset_index(drop=True))
display(weights_after_df.head())

#CSV保存
mapping_path = output_dir / "mesh_station_mapping.csv"
overlap_path = output_dir / "donor_overlap_metadata.csv"
comparison_path = output_dir / "scm_comparison.csv"
series_path = output_dir / "scm_series.csv"
weights_before_path = output_dir / "weights_before_exclusion.csv"
weights_after_path = output_dir / "weights_after_exclusion.csv"
mesh_station_mapping.to_csv(mapping_path,index=False)
donor_metadata.to_csv(overlap_path,index=False)
comparison_df.to_csv(comparison_path,index=False)
series_df.to_csv(series_path,index_label="date")
weights_before_df.to_csv(weights_before_path,index=False)
weights_after_df.to_csv(weights_after_path,index=False)

Number of all meshes: 36
Number of treated meshes: 6
Number of control meshes: 30


,model,donor_count,pre_rmspe,post_rmspe,post_pre_rmspe_ratio,mean_post_gap,true_effect,estimation_error
0,All donors,30,9.043986,222.525067,24.604755,222.308901,220.0,2.308901
1,High-overlap donors excluded,18,12.067109,222.373184,18.428041,222.085552,220.0,2.085552


,actual_treated_area,synthetic_all_donors,synthetic_overlap_excluded,gap_all_donors,gap_overlap_excluded
date,,,,,
2024-01-01,1269.032359,1274.327739,1257.680963,-5.295380,11.351396
2024-01-02,1289.206883,1286.982157,1273.919311,2.224727,15.287572
2024-01-03,1305.904895,1298.944127,1284.310272,6.960768,21.594623
2024-01-04,1316.002988,1310.357281,1292.264617,5.645707,23.738371
2024-01-05,1338.300240,1323.250608,1313.713389,15.049632,24.586851


,h3_id,weight,station,overlap_score,excluded_high_overlap
0,h3_mesh_10,0.114329,ikebukuro,0.75,True
1,h3_mesh_08,0.086828,ikebukuro,0.75,True
2,h3_mesh_09,0.083438,ikebukuro,0.75,True
3,h3_mesh_34,0.075835,kichijoji,0.65,True
4,h3_mesh_28,0.061930,omiya,0.20,False


,h3_id,weight,station,overlap_score
0,h3_mesh_29,0.212197,omiya,0.20
1,h3_mesh_27,0.177528,omiya,0.20
2,h3_mesh_30,0.150233,omiya,0.20
3,h3_mesh_18,0.138482,ueno,0.45
4,h3_mesh_19,0.120682,kawasaki,0.25
